In [ ]:
#input files. Do not touch.
import pandas as pd
import glob
import re

# Load training.csv
training = pd.read_csv("training.csv")

# Load all evalN.csv files into separate dataframes named evaluationN
eval_files = glob.glob("eval*.csv")
for f in eval_files:
    match = re.search(r'eval(\d+)\.csv', f)
    if match:
        n = match.group(1)
        df_name = f"evaluation{n}"
        globals()[df_name] = pd.read_csv(f)
        print("found evaluation dataset n°: " + df_name)

In [ ]:
import numpy as np
import pandas as pd


## PARAMETERS
_WINDOW_SEC       = 0.5   # sliding window width for frame-count rate check (in seconds)
_RATE_K           = 6.0   # σ multiplier for the rate anomaly threshold (6σ is a standard anomaly detection convention)
_RATE_CV_FLOOR    = 0.03  # minimum coefficient of variation for the rate std
_STRICT_IAT_FRAC  = 0.25  # fraction of mean IAT below which a frame is always flagged (to limit FN due to jitter)
_DELTA_MARGIN     = 1.1   # multiplicative margin applied to the per-byte step bound
_DELTA_ADDITIVE   = 7.0   # additive buffer (LSB) applied to the per-byte step bound


# Convert a hex payload string to a list of byte integers
# Returns None if the string has odd length or is empty
def _hex_to_bytes(hex_str):
    s = str(hex_str)
    if len(s) == 0 or len(s) % 2 != 0:  return None
    return [int(s[i:i + 2], 16) for i in range(0, len(s), 2)]


# Returns a dict of "profiles" with per-ID statistics taken from training.csv
# Said statistics are:
# "known_ids"       – Set of arbitration IDs seen in training.
# "iat_stats"       – Per-ID IAT mean (used for strict-IAT supplement).
# "rate_profile"    – Per-ID rolling-window frame-count {mean, std} used by primary rate-anomaly rule.
# "data_len_stats"  – Per-ID payload-length {min, max} for length-check rule.
# "modal_len", "delta_thresh", "zero_thresh_mask"  – Per-ID, per-byte data for payload-step anomaly rule.

def build_profile(train_df):
    W = _WINDOW_SEC
    df = train_df.sort_values("timestamp").copy()
 
    # IAT statistics (for strict-IAT supplement)
    df["iat"] = df.groupby("arbitration_id")["timestamp"].diff()
    iat_stats = (
        df.groupby("arbitration_id")["iat"]
        .agg(["mean", "std"])
        .rename(columns={"mean": "iat_mean", "std": "iat_std"})
    )
 
    # Rolling-window frame-count rate profile
    # For each ID, count how many of its own frames fall in a trailing W-second window at each sample point.
    # The distribution of those counts over the training sequence gives the reference {mean, std}.
    rate_profile = {}
    for arb_id, g in df.groupby("arbitration_id"):
        times = np.sort(g["timestamp"].values)
        n = len(times)
        if n < 2:
            continue
        lo = np.searchsorted(times, times - W, side="left")
        counts = np.arange(1, n + 1) - lo     # frames in (t-W, t]
        stable = (times - times[0]) >= W      # skip the ramp-up period
        if stable.sum() >= 10:
            sc = counts[stable]
            mu = sc.mean()
            sd = sc.std(ddof=0)
            rate_profile[arb_id] = {
                "mean": mu,
                "std": max(sd, mu * _RATE_CV_FLOOR),
            }
        elif arb_id in iat_stats.index:
            # Fewer than one full window of training data: derive from IAT
            mu_i = iat_stats.loc[arb_id, "iat_mean"]
            si_i = iat_stats.loc[arb_id, "iat_std"]
            if mu_i and mu_i > 0:
                en = W / mu_i
                sn = (np.sqrt(en) * si_i / mu_i) if si_i else 0.0
                rate_profile[arb_id] = {
                    "mean": en,
                    "std": max(sn, en * _RATE_CV_FLOOR),
                }
 
    # ── Known IDs and payload-length statistics ──────────────────────────────
    known_ids = set(train_df["arbitration_id"].unique())
    df["data_len"] = train_df["data_field"].str.len()
    data_len_stats = (
        df.groupby("arbitration_id")["data_len"]
        .agg(["mean", "std", "min", "max"])
    )
 
    # ── Per-ID, per-byte step-size model (masquerade detector) ──────────────
    modal_len = {}
    delta_thresh = {}
    zero_thresh_mask = {}   # True for byte positions with zero variance in training
 
    for arb_id, g in df.groupby("arbitration_id"):
        g = g.sort_values("timestamp")
        lens = g["data_field"].str.len()
        if len(lens) == 0:
            continue
        mlen = lens.mode().iloc[0]
        modal_len[arb_id] = mlen
 
        gg = g[lens == mlen]
        if mlen > 0 and mlen % 2 == 0 and len(gg) >= 2:
            byte_df = gg["data_field"].apply(_hex_to_bytes).apply(pd.Series)
            deltas = byte_df.diff().abs().iloc[1:]
            d_max  = deltas.max().values
            d_mean = deltas.mean().values
            d_std  = deltas.std(ddof=0).fillna(0).values
            dt = np.maximum(d_max, d_mean + 6 * d_std)
            delta_thresh[arb_id] = dt
            # Mark byte positions that never varied in training.
            # These are excluded from the step check at detection time to avoid
            # false positives on CAN-multiplexed frames or constant padding bytes
            # that happen to differ between operational sub-modes.
            zero_thresh_mask[arb_id] = dt == 0
        else:
            delta_thresh[arb_id] = None
 
    return {
        "known_ids":        known_ids,
        "iat_stats":        iat_stats,
        "rate_profile":     rate_profile,
        "data_len_stats":   data_len_stats,
        "modal_len":        modal_len,
        "delta_thresh":     delta_thresh,
        "zero_thresh_mask": zero_thresh_mask,
    }
 
 
PROFILE = build_profile(training)
 
 
# ── Detection logic ────────────────────────────────────────────────────────
 
def ids(eval_df):
    """
    Classify each packet in eval_df as attack (1) or benign (0).
 
    Rules
    -----
    1. Unknown arbitration ID → attack
       Any frame whose ID was never seen in training is fabricated (fuzzy or
       DoS injection with a spoofed/new ID).
 
    2a. Rolling-window frame-count rate anomaly → attack
        Count frames of the same ID in the 0.5 s window ending at each frame.
        Flag if the count exceeds the trained mean + 6 σ (floored at 3 % CV).
        Unlike an instantaneous IAT threshold this is insensitive to single-
        frame bunching caused by bus arbitration delays on non-attacked IDs.
 
    2b. Strict IAT supplement → attack
        Separately flag any frame with IAT < 25 % of the trained mean period.
        This catches the very first frames of a flooding burst, which arrive
        before the rolling window has accumulated enough extra frames to exceed
        the count threshold.
 
    3. Data-field length mismatch → attack
       Each ID transmits a consistent payload length.  A different length
       indicates a fuzzy attack or a malformed injection.
 
    4. Implausible payload step (masquerade detector) → attack
       Physical signals change continuously; a CAN frame cannot jump further
       between consecutive same-ID frames than training ever showed.  For each
       ID we track the last verified frame and flag the current frame if any
       byte moves beyond (thresh * 1.1 + 7).  Byte positions whose training
       delta is exactly zero (constant in all training frames, often mux
       indicator or padding bytes) are excluded from this check to avoid false
       positives on frames from operational sub-modes not represented in
       training.
    """
    profile           = PROFILE
    known_ids         = profile["known_ids"]
    iat_stats         = profile["iat_stats"]
    rate_profile      = profile["rate_profile"]
    data_len_stats    = profile["data_len_stats"]
    modal_len         = profile["modal_len"]
    delta_thresh      = profile["delta_thresh"]
    zero_thresh_mask  = profile["zero_thresh_mask"]
    W                 = _WINDOW_SEC
 
    df = eval_df.sort_values("timestamp").copy()
    df["predicted_attack"] = 0
 
    # ── Rule 1: unknown arbitration ID ──────────────────────────────────────
    unknown_mask = ~df["arbitration_id"].isin(known_ids)
    df.loc[unknown_mask, "predicted_attack"] = 1
 
    # ── Rule 2a: rolling-window count-rate anomaly ───────────────────────────
    for arb_id in known_ids:
        if arb_id not in rate_profile:
            continue
        rs        = rate_profile[arb_id]
        threshold = rs["mean"] + _RATE_K * rs["std"]
        mask_id   = df["arbitration_id"] == arb_id
        if not mask_id.any():
            continue
        sub   = df.loc[mask_id].sort_values("timestamp")
        times = sub["timestamp"].values
        lo     = np.searchsorted(times, times - W, side="left")
        counts = np.arange(1, len(times) + 1) - lo
        df.loc[sub.index[counts > threshold], "predicted_attack"] = 1
 
    # ── Rule 2b: strict IAT supplement ──────────────────────────────────────
    df["iat"] = df.groupby("arbitration_id")["timestamp"].diff()
    for arb_id in known_ids:
        if arb_id not in iat_stats.index:
            continue
        iat_mean   = iat_stats.loc[arb_id, "iat_mean"]
        strict_thr = iat_mean * _STRICT_IAT_FRAC
        mask = (
            (df["arbitration_id"] == arb_id)
            & df["iat"].notna()
            & (df["iat"] < strict_thr)
        )
        df.loc[mask, "predicted_attack"] = 1
 
    # ── Rule 3: data-field length mismatch ──────────────────────────────────
    df["data_len"] = df["data_field"].str.len()
    for arb_id, row in data_len_stats.iterrows():
        if arb_id not in known_ids:
            continue
        mask = (
            (df["arbitration_id"] == arb_id)
            & (
                (df["data_len"] < row["min"])
                | (df["data_len"] > row["max"])
            )
        )
        df.loc[mask, "predicted_attack"] = 1
 
    # ── Rule 4: implausible payload step (masquerade detector) ──────────────
    rule4_flag = np.zeros(len(df), dtype=int)
    df_reset   = df.reset_index()
 
    for arb_id, g in df_reset.groupby("arbitration_id"):
        if arb_id not in known_ids:
            continue
        thresh = delta_thresh.get(arb_id)
        mlen   = modal_len.get(arb_id)
        if thresh is None or mlen is None:
            continue
 
        g    = g.sort_values("timestamp")
        sub  = g[g["data_field"].str.len() == mlen]
        if len(sub) < 2:
            continue
 
        byte_vals = (
            sub["data_field"].apply(_hex_to_bytes).apply(pd.Series).values.astype(float)
        )
 
        # Per-byte bound: trained step * margin + additive buffer.
        # Bytes with zero training variance get an infinite bound (excluded).
        bound = np.array(thresh) * _DELTA_MARGIN + _DELTA_ADDITIVE
        if arb_id in zero_thresh_mask:
            bound[zero_thresh_mask[arb_id]] = np.inf
 
        last_trusted = byte_vals[0]
        local_flags  = np.zeros(len(sub), dtype=int)
 
        for i in range(1, len(sub)):
            diff = np.abs(byte_vals[i] - last_trusted)
            if np.any(diff > bound):
                local_flags[i] = 1
                # Keep the anchor frozen at the last verified frame so that a
                # sustained forged payload continues to trip the rule.
            else:
                last_trusted = byte_vals[i]
 
        rule4_flag[sub["index"].values] = local_flags
 
    df["predicted_attack"] = np.maximum(df["predicted_attack"].values, rule4_flag)
 
    # Restore original row order and drop helper columns
    result_df = df.sort_index().drop(columns=["iat", "data_len"])
    return result_df

In [ ]:
#evaluation sequence. Do not touch.
import time  # Add this
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

start_time = time.time()  # Start timing

evaluation_metrics = {}
all_globals = list(globals().items())

# Loop through evaluation dataframes
for name, df in all_globals:
    if name.startswith("evaluation") and isinstance(df, pd.DataFrame):
        # Run IDS
        print("running " + name)
        detected_df = ids(df)
        
        # Compare predictions to ground truth
        y_true = detected_df['attack']
        y_pred = detected_df['predicted_attack']
        
        # Compute metrics
        metrics = {
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1_score': f1_score(y_true, y_pred, zero_division=0),
        }
        
        evaluation_metrics[name] = metrics

# Compute average metrics
avg_metrics = {
    metric: np.mean([metrics[metric] for metrics in evaluation_metrics.values()])
    for metric in ['accuracy', 'precision', 'recall', 'f1_score']
}
evaluation_metrics['Average'] = avg_metrics

# Display metrics
for name, metrics in evaluation_metrics.items():
    print(f"\n{name} Evaluation Metrics:")
    for metric, value in metrics.items():
        print(f"  {metric.capitalize()}: {value:.4f}")

# Plotting
labels = list(evaluation_metrics.keys())
x = np.arange(len(labels))
width = 0.2

# Prepare metric lists
accuracy = [evaluation_metrics[label]['accuracy'] for label in labels]
precision = [evaluation_metrics[label]['precision'] for label in labels]
recall = [evaluation_metrics[label]['recall'] for label in labels]
f1 = [evaluation_metrics[label]['f1_score'] for label in labels]

# Create the bar chart
plt.figure(figsize=(12, 6))
plt.bar(x - 1.5*width, accuracy, width, label='Accuracy')
plt.bar(x - 0.5*width, precision, width, label='Precision')
plt.bar(x + 0.5*width, recall, width, label='Recall')
plt.bar(x + 1.5*width, f1, width, label='F1 Score')

plt.xticks(x, labels, rotation=45)
plt.ylabel('Score')
plt.title('IDS Evaluation Metrics per Dataset')
plt.ylim(0, 1.05)
plt.legend()
plt.tight_layout()
plt.grid(axis='y')
plt.show()

# End timing
end_time = time.time()
elapsed_time = end_time - start_time
print(f"\nTotal Elapsed Time: {elapsed_time:.2f} seconds")